# Week 13 - Dynamic Vision: Motion, Optical Flow and Tracking

**MCTE 4323 / MCTA 4364 Machine Vision**

### Learning objectives
By the end of this lab you will be able to:
- Detect motion with **frame differencing** and **background subtraction**.
- Compute **sparse** (Lucas-Kanade) and **dense** (Farneback) **optical flow**.
- Track multiple objects and count them crossing a line.
- Choose between classical motion detection and **learned tracking** (ByteTrack/DeepSORT).

### Motion cues
A video adds the **time** dimension. Motion can be measured by comparing frames, by modelling the static background, or by estimating the **optical flow** field (apparent pixel motion).

## 1. Setup

In [ ]:
import os
if not os.path.isdir("MCTA-4364-Machine-Vision"):
    !git clone https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git
%cd MCTA-4364-Machine-Vision
!pip -q install opencv-python matplotlib numpy ipywidgets

In [ ]:
import sys
sys.path.append("resources/scripts")
import cv2, numpy as np
from cvhelpers import show, concept_map
print("OpenCV:", cv2.__version__)

## 2. The motion pipeline

In [ ]:
concept_map([
    "Video frames f(t), f(t+1), ...",
    "Motion detection: frame differencing / background subtraction",
    "Optical flow: sparse (Lucas-Kanade) or dense (Farneback)",
    "Association: match detections across frames (IoU / appearance)",
    "Tracking: Kalman filter + Hungarian assignment (ByteTrack/DeepSORT)",
    "Event logic: line crossing, dwell time, speed"
], title="Dynamic vision pipeline")

## 3. Guided example - synthesise a moving scene
We create frames with a moving circle and a moving square over a static textured background. This lets us test every algorithm reproducibly.

In [ ]:
W, H, N = 480, 320, 40

def make_frame(t):
    rng = np.random.RandomState(1)
    frame = (rng.rand(H, W) * 40 + 90).astype(np.uint8)   # static noisy background
    frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
    cx = int(40 + t * 8)
    cv2.circle(frame, (cx, 100), 20, (0, 0, 255), -1)          # red circle moves right
    cy = int(70 + t * 4)
    cv2.rectangle(frame, (330, cy), (410, cy + 60), (0, 255, 0), -1)  # green square moves down
    return frame

frames = [make_frame(t) for t in range(N)]
show(frames[0], frames[15], frames[30], titles=["t=0", "t=15", "t=30"])

## 4. Guided example - frame differencing vs background subtraction
- **Frame differencing**: $|f_t - f_{t-1}|$ finds what changed recently.
- **Background subtraction (MOG2)**: builds a statistical background model and finds what deviates from it.

In [ ]:
prev = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
diff = cv2.absdiff(cv2.cvtColor(frames[1], cv2.COLOR_BGR2GRAY), prev)
_, diff_mask = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)

mog = cv2.createBackgroundSubtractorMOG2(history=200, varThreshold=25, detectShadows=True)
for f in frames:
    fg = mog.apply(f)
_, fg_mask = cv2.threshold(fg, 200, 255, cv2.THRESH_BINARY)

show(diff_mask, fg_mask, titles=["Frame differencing", "MOG2 background subtraction"])

###  Interactive exploration - differencing threshold
A low threshold catches subtle motion but also noise; a high threshold is robust but may miss slow objects.

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

def diff_demo(threshold=25, t=20):
    g = cv2.cvtColor(frames[t], cv2.COLOR_BGR2GRAY)
    d = cv2.absdiff(g, cv2.cvtColor(frames[t - 1], cv2.COLOR_BGR2GRAY))
    _, m = cv2.threshold(d, threshold, 255, cv2.THRESH_BINARY)
    show(m, titles=[f"|f(t) - f(t-1)| > {threshold}"])

interact(diff_demo,
         threshold=widgets.IntSlider(min=5, max=80, step=5, value=25),
         t=widgets.IntSlider(min=1, max=N - 1, step=1, value=20))

## 5. Guided example - sparse optical flow (Lucas-Kanade)
We pick strong corners in one frame and estimate their displacement to the next frame. Arrows show the **motion vectors**.

In [ ]:
g0 = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
g1 = cv2.cvtColor(frames[1], cv2.COLOR_BGR2GRAY)
p0 = cv2.goodFeaturesToTrack(g0, maxCorners=80, qualityLevel=0.01, minDistance=10)
p1, st, err = cv2.calcOpticalFlowPyrLK(g0, g1, p0, None)

good0 = p0[st == 1]
good1 = p1[st == 1]
vis = frames[0].copy()
for (x0, y0), (x1, y1) in zip(good0.reshape(-1, 2), good1.reshape(-1, 2)):
    cv2.arrowedLine(vis, (int(x0), int(y0)), (int(x1), int(y1)), (255, 0, 0), 1, tipLength=0.3)
print("Tracked points:", len(good0))
show(vis, titles=["Lucas-Kanade sparse flow (frame 0 -> 1)"])

## 6. Guided example - dense optical flow (Farneback)
Farneback estimates a flow vector for **every pixel**. We encode direction as hue and magnitude as brightness.

In [ ]:
flow = cv2.calcOpticalFlowFarneback(g0, g1, None, 0.5, 3, 15, 3, 5, 1.2, 0)
mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
hsv = np.zeros((H, W, 3), np.uint8)
hsv[..., 0] = ang * 180 / np.pi / 2      # direction -> hue
hsv[..., 1] = 255
hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)   # magnitude -> brightness
rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
show(rgb, titles=["Farneback dense flow (hue=direction, brightness=speed)"])

## 7. Guided example - tracking by centroid + line counting
A simple tracker assigns IDs to moving blobs and counts them when they cross a line. Industry uses the same *event logic* with stronger trackers.

In [ ]:
tracker = cv2.createBackgroundSubtractorMOG2(history=200, varThreshold=25, detectShadows=False)
LINE_X = 240
counted = set()
count = 0

for t, f in enumerate(frames):
    fg = tracker.apply(f)
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    n, labels, stats, cents = cv2.connectedComponentsWithStats(fg)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] < 150:
            continue
        cx, cy = cents[i]
        if cx > LINE_X and i not in counted:
            counted.add(i)
            count += 1
print("Objects that crossed x =", LINE_X, ":", count)

## 8. Learned tracking (YOLO + ByteTrack) - template
For real scenes, detections come from a detector and tracking is done by **ByteTrack** or **DeepSORT**, which handle occlusion and identity switches far better than centroid tracking.

```python
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
results = model.track(source='clip.mp4', tracker='bytetrack.yaml', persist=True, stream=True)
for r in results:
    boxes = r.boxes
    if boxes.id is not None:
        ids = boxes.id.int().tolist()
        # count ids crossing a line
```

The repository contains a working vehicle counter at `notebooks/13_motion_tracking/count_cars.py`.

In [ ]:
# Optional - run on an uploaded clip named 'clip.mp4'
import os
if os.path.exists("clip.mp4"):
    try:
        from ultralytics import YOLO
        model = YOLO("yolo11n.pt")
        r = next(iter(model.track(source="clip.mp4", tracker="bytetrack.yaml",
                                  persist=True, stream=True, conf=0.3)))
        show(r.plot(), titles=["YOLO + ByteTrack (first frame with IDs)"])
    except Exception as e:
        print("Tracker demo skipped:", e)
else:
    print("Upload clip.mp4 to run the learned tracker.")

## 9. Exercise (complete the code)

1. Modify `make_frame` so the red circle moves **left to right** and count crossings of a vertical line at `x = 240`.
2. Add a **speed estimate**: compute each blob centroid displacement per frame and convert to px/frame.
3. Compare frame differencing and MOG2 for a **slowly moving** object. Which detects it more reliably?

In [ ]:
# TODO: tracking and speed estimation


## 10. Challenge (independent)

Design a **traffic analytics** pipeline: YOLO + ByteTrack on a road video, count vehicles per class crossing a line, and estimate average speed (assume a known real-world distance between two lines). List the failure cases (occlusion, parked cars, camera shake).

In [ ]:
# Your code here


## 11. Check your understanding (Q&A)

<details><summary><b>Q1. When would you prefer background subtraction over frame differencing?</b></summary>

When the camera is static and you need to detect <em>all</em> objects currently present, including stationary-in-image objects. Frame differencing only detects change between two frames, so a temporarily stopped object can vanish.
</details>

<details><summary><b>Q2. What does the brightness of a Farneback flow image represent?</b></summary>

The magnitude (speed) of the apparent motion. Hue represents its direction.
</details>

<details><summary><b>Q3. Why is optical flow not the same as true motion?</b></summary>

Optical flow is the <em>apparent</em> pixel motion. Textureless regions, lighting changes and the aperture problem can make it differ from the real 3D motion.
</details>

<details><summary><b>Q4. Why do learned trackers use both motion and appearance?</b></summary>

Motion (Kalman filter) predicts where an object goes; appearance (deep features) re-identifies it after occlusion. Together they reduce identity switches.
</details>

## 12. Further reading & self-exploration
- OpenCV optical flow: https://docs.opencv.org/4.x/d4/dee/tutorial_optical_flow.html
- OpenCV background subtraction: https://docs.opencv.org/4.x/d1/dc5/tutorial_background_subtraction.html
- Ultralytics track mode (ByteTrack/BoT-SORT): https://docs.ultralytics.com/modes/track/
- Simple Online and Realtime Tracking (SORT) and ByteTrack papers.
- Wikipedia - Optical flow: https://en.wikipedia.org/wiki/Optical_flow
- Wikipedia - Video tracking: https://en.wikipedia.org/wiki/Video_tracking

**Try next:** add a Kalman filter to your centroid tracker and compare ID stability.

## 13. Key takeaways
- Motion: frame differencing, background subtraction, optical flow.
- Sparse flow tracks features; dense flow gives a full motion field.
- Tracking = prediction + association across frames.
- Learned trackers (ByteTrack/DeepSORT) handle real-world occlusion.
- Event logic (line crossing, speed) turns tracking into analytics.